In [18]:
import tensorflow as tf
import tf2onnx
import os

# 1. بناء الهيكلية العينية لموديلك (نفس الموجود في num_model.ipynb)
def build_exact_model():
    model = tf.keras.Sequential([
        # الطبقة الأولى LSTM (64 units)
        tf.keras.layers.LSTM(64, return_sequences=True, activation='relu', input_shape=(90, 42)),
        # الطبقة الثانية LSTM (128 units)
        tf.keras.layers.LSTM(128, return_sequences=True, activation='relu'),
        # الطبقة الثالثة LSTM (64 units)
        tf.keras.layers.LSTM(64, return_sequences=False, activation='relu'),
        # الطبقات المتصلة Dense
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(32, activation='relu'),
        # الطبقة الأخيرة (عدد الكلاسات عندك هو 4 بناءً على الكود)
        tf.keras.layers.Dense(4, activation='softmax')
    ])
    return model

h5_path = "sign_language_model.h5"
onnx_path = "sign_language_model.onnx"

if not os.path.exists(h5_path):
    print(f"❌ Error: {h5_path} non found!")
else:
    print("🛠️ Rebuilding model architecture...")
    model = build_exact_model()

    try:
        print("⚖️ Loading weights...")
        # تحميل الأوزان فقط يتخطى أخطاء توافق الإصدارات تماماً
        model.load_weights(h5_path)
        print("✅ Weights loaded successfully!")

        # 2. التحويل إلى ONNX
        print("🚀 Converting to ONNX format...")
        spec = (tf.TensorSpec((None, 90, 42), tf.float32, name="input"),)
        
        # تحويل الموديل
        model_proto, _ = tf2onnx.convert.from_keras(
            model, 
            input_signature=spec, 
            opset=13, 
            output_path=onnx_path
        )
        print(f"🎉 SUCCESS! File saved: {onnx_path}")
        
    except Exception as e:
        print(f"❌ Failed: {e}")

🛠️ Rebuilding model architecture...
⚖️ Loading weights...
❌ Failed: Layer count mismatch when loading weights from file. Model expected 6 layers, found 0 saved layers.


In [14]:
import os
import tensorflow as tf
# استيراد محرك كيراس المتوافق مع الإصدارات القديمة
import tf_keras as keras 
import tf2onnx

h5_model_name = "numbers_model.h5" 
onnx_model_name = "sign_language_model.onnx"

print("Loading model using tf_keras (Legacy Mode)...")

try:
    # استخدام keras (المستوردة من tf_keras) بدلاً من tf.keras
    model = keras.models.load_model(h5_model_name, compile=False)
    
    print("Model loaded successfully! Starting ONNX conversion...")
    
    # بناءً على الصورة الـ Input Shape عندك هو (90, 42)
    spec = (tf.TensorSpec((None, 90, 42), tf.float32, name="input"),)

    # التحويل
    model_proto, _ = tf2onnx.convert.from_keras(
        model, 
        input_signature=spec, 
        opset=13, 
        output_path=onnx_model_name
    )
    
    print(f"DONE! Model saved as: {onnx_model_name}")

except Exception as e:
    print(f"An error occurred: {e}")

ModuleNotFoundError: No module named 'tf_keras'

In [ ]:
python convert_h5_to_onnx.py


In [ ]:
python -m pip install onnxruntime


In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession("model.onnx")
print([i.name for i in sess.get_inputs()])
print([o.name for o in sess.get_outputs()])
